# Factuality Degradation in Iterative LLM Rewriting
**Baseline pipeline (no Self-Refine) — phased execution**

Thesis project — Anna Sacchet

This notebook implements the baseline rewriting pipeline (Section 5.3).
Execution is split into **independent phases** so that each can run, crash,
and resume without losing work from earlier phases:

| Phase | What runs | Output file | Needs GPU? |
|-------|-----------|-------------|------------|
| 1 | Rewriting chains E₀→E₁→E₂→E₃ | `rewriting_chains.jsonl` | Yes |
| 2 | BERTScore + length (offline) | `metrics_offline.jsonl` | Yes (BERTScore model) |
| 3 | QA inference | `qa_predictions.jsonl` | Yes |
| 4 | FactScore (claim decomposition + verification) | `factscore_results.jsonl` | Yes |
| 5 | Assembly → P/A/U aggregation + plots | `results_merged.jsonl`, CSVs | No |

Self-Refine (Section 5.4) lives in a separate notebook.

## 1 · Setup

In [ ]:
# Run once in a fresh environment
# !pip install -q transformers accelerate bert-score pandas tqdm matplotlib

In [ ]:
import json
import os
import random
import re
import string
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

In [ ]:
from google.colab import files

uploaded = files.upload()

# Per vedere i file caricati
for filename in uploaded.keys():
    print('File caricato:', filename)

## 2 · Configuration

In [ ]:
CONFIG = {
    # --- Data ---
    "dataset_path": "musique_ans_v1.0_dev.jsonl",
    "output_dir": "results",

    # --- Models ---
    "rewriter_model": "allenai/OLMo-2-0325-32B-Instruct",
    "qa_model":       "allenai/OLMo-2-32B-Instruct",
    "claim_model":    "allenai/OLMo-2-32B-Instruct",

    # --- Experiment ---
    "n_per_hop": 405,             # balanced across 2/3/4-hop
    "n_runs": 5,                  # >= 5 per P/A/U framework
    "n_iterations": 3,            # E0 -> E1 -> E2 -> E3
    "rewrite_temperature": 0.7,
    "qa_temperature": 0.0,        # deterministic QA
    "max_new_tokens_rewrite": 2048,
    "max_new_tokens_qa": 64,
    "max_new_tokens_claims": 512,
    "seed": 42,

    # --- Smoke test ---
    "smoke_test": True,
    "smoke_test_n_per_hop": 3,
}

OUT = Path(CONFIG["output_dir"])
OUT.mkdir(exist_ok=True, parents=True)

# Phase output files
CHAINS_FILE    = OUT / "rewriting_chains.jsonl"
OFFLINE_FILE   = OUT / "metrics_offline.jsonl"
QA_FILE        = OUT / "qa_predictions.jsonl"
FACTSCORE_FILE = OUT / "factscore_results.jsonl"
MERGED_FILE    = OUT / "results_merged.jsonl"

## 3 · Utilities

In [ ]:
def load_jsonl(path):
    """Load a JSONL file, skipping malformed lines."""
    rows = []
    if not Path(path).exists():
        return rows
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    return rows


def append_jsonl(path, record):
    """Append a single record to a JSONL file."""
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def done_keys_from_file(path, key_fields):
    """Build a set of already-completed keys for resume support."""
    rows = load_jsonl(path)
    return {tuple(r[k] for k in key_fields) for r in rows}

## 4 · Dataset — load & balance by hop count

MuSiQue dev: 1252 / 760 / 405 for 2/3/4-hop.
Downsample to 405 per group so conditions are directly comparable.

In [ ]:
def load_musique(path):
    items = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                items.append(json.loads(line))
    return items


def hop_count(item):
    qid = item.get("id", "")
    m = re.match(r"(\d+)hop__", qid)
    if m:
        return int(m.group(1))
    return len(item.get("question_decomposition", []))


def extract_evidence(item):
    """Concatenate supporting paragraphs (with titles) to form E0."""
    support = [p for p in item["paragraphs"] if p.get("is_supporting")]
    support.sort(key=lambda p: p.get("idx", 0))
    return "\n\n".join(f"{p['title']}. {p['paragraph_text']}" for p in support)


def balance_by_hop(items, n_per_hop, seed=42):
    rng = random.Random(seed)
    by_hop = defaultdict(list)
    for it in items:
        by_hop[hop_count(it)].append(it)
    balanced = []
    for h in (2, 3, 4):
        pool = list(by_hop[h])
        rng.shuffle(pool)
        balanced.extend(pool[:min(n_per_hop, len(pool))])
    return balanced

In [ ]:
raw = load_musique(CONFIG["dataset_path"])
print(f"Loaded {len(raw)} raw MuSiQue items")
print("Raw hop distribution:", {h: sum(1 for it in raw if hop_count(it) == h) for h in (2, 3, 4)})

n = CONFIG["smoke_test_n_per_hop"] if CONFIG["smoke_test"] else CONFIG["n_per_hop"]
subset = balance_by_hop(raw, n, seed=CONFIG["seed"])

questions = []
for it in subset:
    questions.append({
        "qid": it["id"],
        "hop": hop_count(it),
        "question": it["question"],
        "gold_answer": it["answer"],
        "gold_aliases": it.get("answer_aliases", []),
        "E0": extract_evidence(it),
    })

print(f"\nUsing {len(questions)} questions")
print("Balanced hop distribution:", {h: sum(1 for q in questions if q['hop'] == h) for h in (2, 3, 4)})

## 5 · Rewriting instructions

From OpenRewriteEval (Shu et al., 2023).
Two style-oriented categories (formality, paraphrase) and two content-oriented (shorten, elaborate).

The **elaborate** instructions use grounded variants that forbid introducing
unsupported content — design choice to revisit (see brainstorming doc, Section 5.2).

In [ ]:
ALL_INSTRUCTIONS = {
    # ---- STYLE-ORIENTED ----
    ("style", "formality"): [
        "Make the text more formal.",
        "Rephrase it to be more formal.",
        "Too conversational, rephrase it to be more formal.",
    ],
    ("style", "paraphrase"): [
        "Paraphrase this.",
        "Reword this text.",
        "Use different wording.",
    ],
    # ---- CONTENT-ORIENTED ----
    ("content", "shorten"): [
        "Make wording more concise.",
        "Rephrase for clarity and conciseness.",
        "Improve accuracy, clarity, and conciseness of language.",
    ],
    ("content", "elaborate"): [
        "Elaborate on the content, adding relevant details while staying faithful to the source text.",
        "Expand the text with more context, without introducing information that is not supported by the original.",
        "Add more detail, keeping every fact grounded in the source material.",
    ],
}

REWRITE_TEMPLATE = """You will rewrite the text below according to the instruction.
Return ONLY the rewritten text, with no preamble or commentary.

Instruction: {instruction}

Text:
{text}

Rewritten text:"""

for key, prompts in ALL_INSTRUCTIONS.items():
    print(key, "->", len(prompts), "variants")

---
# PHASE 1 · Rewriting inference

Generate all rewriting chains and save them. **No metrics computed here.**
This is the most expensive phase — everything downstream can reuse these texts.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

def load_causal_lm(model_id, dtype=torch.bfloat16):
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=dtype,
        device_map="auto",
        trust_remote_code=True,
    )
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model.eval()
    return tok, model


@torch.no_grad()
def generate(tokenizer, model, user_prompt, *, temperature=0.7, max_new_tokens=512, do_sample=True):
    """Single-prompt generation via the model's chat template."""
    messages = [{"role": "user", "content": user_prompt}]
    if getattr(tokenizer, "chat_template", None):
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        text = user_prompt

    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.pad_token_id,
    )
    if do_sample and temperature > 0:
        gen_kwargs.update(do_sample=True, temperature=temperature, top_p=0.95)
    else:
        gen_kwargs.update(do_sample=False)

    out = model.generate(**inputs, **gen_kwargs)
    new_tokens = out[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Set HF_TOKEN in Colab Secrets (sidebar key icon)
login(token=userdata.get("HF_TOKEN"))


In [ ]:
rewriter_tok, rewriter_model = load_causal_lm(CONFIG["rewriter_model"])

In [ ]:
def run_rewriting_chain(E0, instruction_pool, *, n_iterations, temperature, run_seed):
    """Return (chain, instructions_used) where chain = [E0, E1, ..., E_n]."""
    rng = random.Random(run_seed)
    chain = [E0]
    used = []
    current = E0
    for _ in range(n_iterations):
        instr = rng.choice(instruction_pool)  # ← was hardcoded to pool[0]
        used.append(instr)
        prompt = REWRITE_TEMPLATE.format(instruction=instr, text=current)
        current = generate(
            rewriter_tok, rewriter_model, prompt,
            temperature=temperature,
            max_new_tokens=CONFIG["max_new_tokens_rewrite"],
            do_sample=True,
        )
        chain.append(current)
    return chain, used

In [ ]:
# Reload dataset as a flat list for fast id->record lookup
import json

dataset = []
with open(CONFIG["dataset_path"], "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            dataset.append(json.loads(line))

dataset_by_id = {d["id"]: d for d in dataset}

def build_E0_full(qid):
    """Concatenate all paragraphs (not only the supporting ones)."""
    d = dataset_by_id.get(qid)
    if d is None:
        return None
    parts = []
    for p in d["paragraphs"]:
        parts.append(f"{p['title']}. {p['paragraph_text']}")
    return "\n\n".join(parts)


Ultimo funzionante, una domanda, tutte le possibilità stilistiche, con tutte le evidenze

In [ ]:
results = []

for q in questions[:1]:
    E0 = build_E0_full(q["qid"])  # ← era EE0

    for (group, instruction_type), pool in ALL_INSTRUCTIONS.items():
        for run, instr in enumerate(pool):
            key      = (q["qid"], group, instruction_type, run)
            run_seed = hash((*key, CONFIG["seed"])) & 0xFFFFFFFF

            chain, used = run_rewriting_chain(
                E0,                               # ← ora usa il valore giusto
                [instr],
                n_iterations=CONFIG["n_iterations"],
                temperature=CONFIG["rewrite_temperature"],
                run_seed=run_seed,
            )

            results.append({
                "qid":              q["qid"],
                "question":         q["question"],
                "group":            group,
                "instruction_type": instruction_type,
                "run":              run,
                "run_seed":         run_seed,
                "instruction_used": used,
                "chain": [
                    {"step": i, "text": step}
                    for i, step in enumerate(chain)
                ],
            })

print(f"Total chains generated: {len(results)}")

In [ ]:
import pandas as pd

rows = []
for r in results:
    for step in r['chain']:
        rows.append({
            "qid":              r['qid'],
            "question":         r['question'],
            "group":            r['group'],
            "instruction_type": r['instruction_type'],
            "run":              r['run'],
            "instruction_used": r['instruction_used'][step['step']] if step['step'] < len(r['instruction_used']) else "",
            "step":             step['step'],
            "text":             step['text'],
        })

df = pd.DataFrame(rows)
df.to_csv("results/rewriting_chains.csv", index=False, encoding="utf-8")
print(f"Salvato: {len(df)} righe → results/rewriting_chains.csv")

In [ ]:
from google.colab import files
files.download("results/rewriting_chains.csv")

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()

CHAINS_CSV = None
MUSIQUE_PATH = None

for filename in uploaded.keys():
    print('File caricato:', filename)
    if 'chain' in filename.lower():
        CHAINS_CSV = Path("/content/" + filename)
    elif 'musique' in filename.lower():
        MUSIQUE_PATH = Path("/content/" + filename)

if CHAINS_CSV is None:
    print("Carica il file chains CSV")
    up = files.upload()
    CHAINS_CSV = Path("/content/" + list(up.keys())[0])

if MUSIQUE_PATH is None:
    print("Carica il file MuSiQue JSONL")
    up = files.upload()
    MUSIQUE_PATH = Path("/content/" + list(up.keys())[0])

OUTPUT_CSV = Path("/content/rewriting_chains32b_answer_f1.csv")

print(f"\n✅ chains:  {CHAINS_CSV}")
print(f"✅ musique: {MUSIQUE_PATH}")

In [ ]:
# ============================================================
# CELLA 1 — Installa dipendenze
# ============================================================
# Esegui questa cella per prima cosa (Runtime > Run cell)
# !pip install -q transformers accelerate pandas torch

# ============================================================
# CELLA 2 — Upload dei file e configurazione
# ============================================================

import os
import io
import json
import re
import string
import time
from collections import Counter
from pathlib import Path

import pandas as pd

# ------------------------------------------------------------------
# Upload interattivo in Colab
# ------------------------------------------------------------------
# Decommentare questo blocco SE si vuole caricare i file manualmente
# ad ogni esecuzione (apparirà il tasto "Scegli file"):
#
# from google.colab import files
# print("Carica rewriting_chains32b.csv")
# up1 = files.upload()
# print("Carica musique_ans_v1.0_dev.jsonl")
# up2 = files.upload()
#
# I file vengono salvati automaticamente in /content/ da Colab.
# ------------------------------------------------------------------

# Posizione dei file dopo l'upload (Colab li salva sempre in /content/)
CHAINS_CSV   = Path("/content/rewriting_chains32b.csv")
MUSIQUE_PATH = Path("/content/musique_ans_v1.0_dev.jsonl")

# Output: viene scritto in /content/ (scaricabile con files.download)
OUTPUT_CSV = Path("/content/rewriting_chains32b_answer_f1.csv")

# ------------------------------------------------------------------
# Se hai montato Google Drive puoi usare percorsi Drive invece:
#
# from google.google.drive import drive
# drive.mount('/content/drive')
# CHAINS_CSV   = Path("/content/drive/MyDrive/tuo_percorso/rewriting_chains32b.csv")
# MUSIQUE_PATH = Path("/content/drive/MyDrive/tuo_percorso/musique_ans_v1.0_dev.jsonl")
# OUTPUT_CSV   = Path("/content/drive/MyDrive/tuo_percorso/rewriting_chains32b_answer_f1.csv")
# ------------------------------------------------------------------

def check_files():
    """Verifica che i file siano presenti e mostra info utili."""
    ok = True
    for p in [CHAINS_CSV, MUSIQUE_PATH]:
        if p.exists():
            size_mb = p.stat().st_size / 1_048_576
            print(f"  ✅  {p.name}  ({size_mb:.1f} MB)")
        else:
            print(f"  ❌  {p.name} NON trovato in {p.parent}")
            ok = False
    if not ok:
        raise FileNotFoundError(
            "Carica i file mancanti con:\n"
            "  from google.colab import files; files.upload()\n"
            "oppure monta Drive e aggiorna i percorsi sopra."
        )

print("Verifica file:")
check_files()


# ============================================================
# CELLA 3 — Configurazione esperimento
# ============================================================

QA_MODEL_ID  = "allenai/OLMo-2-1124-13B-Instruct"   # cambia se serve
CHAIN_KEYS   = ["qid", "group", "instruction_type", "run"]

MAX_NEW_TOKENS = 64
TEMPERATURE    = 0.0
BATCH_SIZE     = 4   # riduci a 2 se la GPU va in OOM

# Pilot mode: valuta solo 1 domanda / run 0.
# Metti TEST_MODE = False per valutare l'intero dataset.
TEST_MODE   = True
TEST_FILTER = {"qid": "2hop__635544_110949", "run": 0}

QA_USER_TEMPLATE = """{context}

{question}

Answer with only the essential information (a name, date, number, or short phrase). No explanation, no full sentences."""


# ============================================================
# CELLA 4 — Funzioni di supporto (esegui una volta)
# ============================================================

# ---- MuSiQue loader ----

def load_musique_index(path: Path) -> dict:
    """Restituisce {qid: {'question': str, 'answer': str, 'aliases': [str]}}."""
    index = {}
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            qid = rec["id"]
            index[qid] = {
                "question": rec["question"],
                "answer":   rec["answer"],
                "aliases":  rec.get("answer_aliases") or [],
            }
    return index


# ---- Answer F1 (MuSiQue-official, SQuAD-style) ----

def normalize_answer(s):
    def remove_articles(text):
        return re.sub(r"\b(a|an|the)\b", " ", text, flags=re.UNICODE)
    def white_space_fix(text):
        return " ".join(text.split())
    def remove_punc(text):
        exclude = set(string.punctuation)
        return "".join(ch for ch in text if ch not in exclude)
    return white_space_fix(remove_articles(remove_punc(s.lower())))

def get_tokens(s):
    return normalize_answer(s).split() if s else []

def compute_f1(a_gold, a_pred):
    gold_toks = get_tokens(a_gold)
    pred_toks = get_tokens(a_pred)
    common    = Counter(gold_toks) & Counter(pred_toks)
    num_same  = sum(common.values())
    if not gold_toks or not pred_toks:
        return int(gold_toks == pred_toks)
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_toks)
    recall    = num_same / len(gold_toks)
    return (2 * precision * recall) / (precision + recall)

def best_f1(pred, gold, aliases):
    """F1 massima su {gold} ∪ aliases, come in evaluate_v1.0.py."""
    candidates = [gold] + [a for a in aliases if a]
    best, best_ref = 0.0, gold
    for ref in candidates:
        s = compute_f1(ref, pred)
        if s > best:
            best, best_ref = s, ref
    return best, best_ref


def extract_short_answer(text: str) -> str:
    """Seconda linea di difesa contro risposte verbose.

    Rimuove frasi di apertura tipo "The answer is...", "Based on the text..."
    e "...is not explicitly mentioned..." poi restituisce solo la prima frase.
    Il prompt gia chiede risposte brevi; questa funzione gestisce i casi
    in cui il modello ignora l'istruzione.
    """
    text = re.sub(
        r"^(the answer is\s*:?\s*|"
        r"based on (the )?(text|passage|context)[,.]?\s*|"
        r"according to (the )?(text|passage)[,.]?\s*|"
        r"the .{0,60} is not explicitly mentioned[^.]*\.\s*)",
        "", text, flags=re.IGNORECASE,
    ).strip()
    first = re.split(r"(?<=[.!?])\s+", text)[0].strip()
    return first if first else text


# ---- Modello OLMo ----

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def load_model(model_id: str):
    print(f"\nCaricamento {model_id} ...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"   # obbligatorio per generazione batched decoder-only

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )
    model.eval()
    device_info = getattr(model, "hf_device_map", "n/a")
    print(f"  device map: {device_info}")
    return tokenizer, model

def build_prompts(tokenizer, rows: list[dict], musique: dict) -> list[str]:
    prompts = []
    for row in rows:
        ref  = musique[row["qid"]]
        user = QA_USER_TEMPLATE.format(
            context=row["text"].strip(),
            question=ref["question"].strip(),
        )
        prompt = tokenizer.apply_chat_template(
            [{"role": "user", "content": user}],
            tokenize=False,
            add_generation_prompt=True,
        )
        prompts.append(prompt)
    return prompts

@torch.no_grad()
def generate_batch(tokenizer, model, prompts: list[str]) -> list[str]:
    enc = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=False,
    ).to(model.device)

    gen_kwargs = dict(
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=(TEMPERATURE > 0.0),
        pad_token_id=tokenizer.pad_token_id,
    )
    if TEMPERATURE > 0.0:
        gen_kwargs["temperature"] = TEMPERATURE

    out        = model.generate(**enc, **gen_kwargs)
    gen_tokens = out[:, enc["input_ids"].shape[1]:]
    texts      = tokenizer.batch_decode(gen_tokens, skip_special_tokens=True)
    return [t.strip() for t in texts]


# ============================================================
# CELLA 5 — Esegui la valutazione
# ============================================================

def run_evaluation():
    # 1. Carica dati
    print("Caricamento MuSiQue index...")
    musique = load_musique_index(MUSIQUE_PATH)
    print(f"  {len(musique)} domande indicizzate")

    df = pd.read_csv(CHAINS_CSV)
    df = df.sort_values(CHAIN_KEYS + ["step"]).reset_index(drop=True)

    missing = set(df["qid"].unique()) - set(musique.keys())
    if missing:
        raise RuntimeError(f"qid nel CSV non trovati in MuSiQue: {missing}")

    # 2. Filtraggio (TEST_MODE)
    to_eval = df.copy()
    if TEST_MODE and TEST_FILTER:
        for k, v in TEST_FILTER.items():
            to_eval = to_eval[to_eval[k] == v]
        print(f"\n*** TEST MODE: filter={TEST_FILTER} → {len(to_eval)} righe ***")

    # 3. Dedup step 0 (stessa E_0 per tutte le instruction_types dello stesso chain)
    e0_mask  = to_eval["step"] == 0
    e0_dedup = to_eval[e0_mask].drop_duplicates(subset=["qid", "run"], keep="first")
    to_eval  = pd.concat([e0_dedup, to_eval[~e0_mask]], ignore_index=True)
    to_eval  = to_eval.sort_values(CHAIN_KEYS + ["step"]).reset_index(drop=True)

    if to_eval.empty:
        raise RuntimeError("Nessuna riga da valutare dopo il filtro.")

    total = len(to_eval)
    print(f"\nAnswer F1 su {total} testi (incluso step 0)  —  QA model = {QA_MODEL_ID}")
    print(f"Batch size: {BATCH_SIZE}\n")

    # 4. Carica modello
    tokenizer, model = load_model(QA_MODEL_ID)

    # 5. Loop di inferenza
    rows    = to_eval.to_dict(orient="records")
    results = []
    t_start = time.time()

    for i in range(0, len(rows), BATCH_SIZE):
        batch   = rows[i : i + BATCH_SIZE]
        prompts = build_prompts(tokenizer, batch, musique)
        preds   = generate_batch(tokenizer, model, prompts)

        for row, pred in zip(batch, preds):
            raw_pred      = pred
            pred          = extract_short_answer(pred)
            ref           = musique[row["qid"]]
            f1, match_ref = best_f1(pred, ref["answer"], ref["aliases"])
            out = {
                **{k: row[k] for k in CHAIN_KEYS},
                "step":                 int(row["step"]),
                "question":             ref["question"],
                "gold_answer":          ref["answer"],
                "raw_predicted_answer": raw_pred,
                "predicted_answer":     pred,
                "matched_reference":    match_ref,
                "answer_f1":            f1,
            }
            results.append(out)
            label      = f"{out['group']}/{out['instruction_type']}/run{out['run']}/step{out['step']}"
            pred_short = (pred[:50] + "...") if len(pred) > 50 else pred
            print(f"[{len(results):>3}/{total}] {label:<55s}  "
                  f"pred={pred_short!r:<55s}  gold={ref['answer']!r:<25s}  F1={f1:.3f}")

    elapsed = time.time() - t_start
    print(f"\nTempo totale: {elapsed:.1f}s ({elapsed/60:.1f} min)")

    # 6. Broadcast E_0 a tutti i (group, instruction_type) dello stesso chain
    results_df = pd.DataFrame(results)
    if not results_df.empty:
        step0    = results_df[results_df["step"] == 0]
        step_gt0 = results_df[results_df["step"] > 0]
        if not step0.empty:
            all_chains = df[CHAIN_KEYS].drop_duplicates()
            if TEST_MODE and TEST_FILTER:
                for k, v in TEST_FILTER.items():
                    all_chains = all_chains[all_chains[k] == v]
            step0_broadcast = all_chains.merge(
                step0.drop(columns=["group", "instruction_type"]),
                on=["qid", "run"],
                how="inner",
            )
            results_df = pd.concat([step0_broadcast, step_gt0], ignore_index=True)
            results_df = results_df.sort_values(CHAIN_KEYS + ["step"]).reset_index(drop=True)

    # 7. Salva (append se il file esiste già)
    if OUTPUT_CSV.exists():
        prev   = pd.read_csv(OUTPUT_CSV)
        merged = pd.concat([prev, results_df], ignore_index=True)
        merged = merged.drop_duplicates(subset=CHAIN_KEYS + ["step"], keep="last")
        merged.to_csv(OUTPUT_CSV, index=False)
    else:
        results_df.to_csv(OUTPUT_CSV, index=False)

    print(f"\nSalvato: {OUTPUT_CSV}")

    # 8. Pivot riepilogativo
    print()
    print("=" * 70)
    print("ANSWER F1 — media per (group, instruction_type, step)")
    print("=" * 70)
    pivot = results_df.pivot_table(
        index=["group", "instruction_type"],
        columns="step",
        values="answer_f1",
        aggfunc="mean",
    )
    print(pivot.round(3))

    # 9. Download automatico del CSV in Colab
    try:
        from google.colab import files
        files.download(str(OUTPUT_CSV))
        print(f"\n📥 Download avviato: {OUTPUT_CSV.name}")
    except ImportError:
        print("\n(Non in Colab: il file è già salvato localmente.)")

    return results_df


# Esegui!
results_df = run_evaluation()